In [0]:
from pyspark.sql.functions import current_timestamp, col

# Rutas gobernadas en Unity Catalog
source_path = "/Volumes/logistics_prod/bronze/landing_zone/"
checkpoint_path = "/Volumes/logistics_prod/bronze/landing_zone/_checkpoints/smart_logistics"
target_table = "logistics_prod.bronze.logistics_raw"

# Lectura incremental con Auto Loader (cloudFiles)
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

# Escritura en tabla Delta Bronze procesando solo lo nuevo disponible
query = (
    df_stream.writeStream
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
print(f"Capa Bronze lista. Total de filas en {target_table}: {spark.table(target_table).count()}")
